# 04 — LOVM-style Baseline vs Router on Test Set

This notebook computes a LOVM-style baseline on the **test set**:

- For each `router_task`, pick the **single best model** (by accuracy) and
  pretend we always use that model for all samples of that task.
- Compare its **accuracy and cost** to our **router**, which picks a model
  per-sample.

Columns assumed (wide format):

- `router_task`
- `router_best_model_name`, `router_chosen_perf`, `router_chosen_cost`
- For each model `m` in `{deepseek_ocr, qwen2_5_vl_3b, qwen2_5_vl_7b,
  qwen3_vl_8b_thinking, gemma_3_27b}`:
  - `m__is_correct`
  - `m__score_f1`
  - `m__sample_score`
  - `m__cost`
  - `m__valid_mask`


In [15]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_rows", 20)
pd.set_option("display.max_columns", 120)


In [8]:
# --- CONFIG: point this to your test eval file ---
DATA_PATH = Path.cwd().parent.parent / "dataset" /"final_dataset" /"router_final"
print(DATA_PATH)
TEST_FILE = DATA_PATH / "router_test_final.parquet"   # or .csv
str(TEST_FILE)

/Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/Which_VLM_Router/dataset/final_dataset/router_final


'/Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/Which_VLM_Router/dataset/final_dataset/router_final/router_test_final.parquet'

### Step-1:- Loading the Dataset

In [9]:
print("Loading test set from:", TEST_FILE)
if TEST_FILE.suffix == ".parquet":
    df_test = pd.read_parquet(TEST_FILE)
else:
    df_test = pd.read_csv(TEST_FILE)

print("Rows:", len(df_test))
print("Columns:", len(df_test.columns))
df_test.head()


Loading test set from: /Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/Which_VLM_Router/dataset/final_dataset/router_final/router_test_final.parquet
Rows: 13707
Columns: 49


,sample_id,image_path,prompt_raw,router_task,source_dataset,source_config,txt_question_type,txt_has_mc_options,img_width,img_height,...,qwen3_vl_8b_thinking__sample_score,qwen3_vl_8b_thinking__cost,qwen3_vl_8b_thinking__valid_mask,qwen3_vl_8b_thinking__is_correct,qwen3_vl_8b_thinking__score_f1,gemma_3_27b__sample_score,gemma_3_27b__cost,gemma_3_27b__valid_mask,gemma_3_27b__is_correct,gemma_3_27b__score_f1
0,ai2d_00004_bf3d9c5fd30bf304,None,Question: If the Termites in the community bel...,diagram_reasoning,cauldron_ai2d,ai2d,None,True,1100.0,532.0,...,1.150000,0.001191,True,True,0.000000,1.150000,0.000060,True,True,0.030303
1,ai2d_00007_e6f58451a22503d1,None,Question: what does the 2nd picture show?\nCho...,diagram_reasoning,cauldron_ai2d,ai2d,None,True,1500.0,1344.0,...,1.150000,0.000758,True,True,0.032258,0.093605,0.000042,True,False,0.046512
2,ai2d_00019_e5f934cfc0951972,None,Question: A food web for a ecosystem is shown ...,diagram_reasoning,cauldron_ai2d,ai2d,None,True,305.0,297.0,...,-0.025000,0.001104,True,False,0.000000,0.104091,0.000059,True,False,0.072727
3,ai2d_00021_59cfa73bc8485386,None,Question: What is the lowermost section of a v...,diagram_reasoning,cauldron_ai2d,ai2d,None,True,1402.0,891.0,...,0.096918,0.000672,True,False,0.054795,1.150000,0.000050,True,True,0.038462
4,ai2d_00023_bd686a44c25987c8,None,Question: What is the insulating envelope of m...,diagram_reasoning,cauldron_ai2d,ai2d,None,True,549.0,284.0,...,0.099615,0.000414,True,False,0.061538,1.200000,0.000030,True,True,0.400000


### Step-2: - Identify the right Column names in our dataset

In [11]:
# Task column (dataset / router task)
TASK_COL = "router_task"   # change if you named it differently
assert TASK_COL in df_test.columns

print("Tasks in test set:")
print(df_test[TASK_COL].value_counts())

Tasks in test set:
router_task
table_reasoning         1822
chart_reasoning         1277
document_ocr             960
general_vqa              915
spatial_reasoning        873
chart_captioning         602
table_math               588
scene_text_ocr           580
geometry_reasoning       504
code_generation          317
medical_report           316
handwriting_ocr          313
difference_detection     313
knowledge_vqa            306
visual_mrc               302
icon_reasoning           301
rendered_text_ocr        296
science_reasoning        291
image_captioning         289
map_reasoning            286
dense_captioning         285
abstract_reasoning       284
counting                 283
diagram_reasoning        282
ui_captioning            281
meme_classification      267
web_understanding        259
textbook_qa              216
medical_vqa               54
diagram_captioning        45
Name: count, dtype: int64


In [13]:
print(df_test.columns)

Index(['sample_id', 'image_path', 'prompt_raw', 'router_task',
       'source_dataset', 'source_config', 'txt_question_type',
       'txt_has_mc_options', 'img_width', 'img_height', 'img_aspect_ratio',
       'txt_prompt_length_chars', 'txt_prompt_length_words', 'ground_truth',
       'ground_truth_type', 'router_best_model_id', 'router_best_model_name',
       'router_chosen_perf', 'router_chosen_cost',
       'router_soft_p_deepseek_ocr', 'router_soft_p_qwen2_5_vl_3b',
       'router_soft_p_qwen2_5_vl_7b', 'router_soft_p_qwen3_vl_8b_thinking',
       'router_soft_p_gemma_3_27b', 'deepseek_ocr__sample_score',
       'deepseek_ocr__cost', 'deepseek_ocr__valid_mask',
       'deepseek_ocr__is_correct', 'deepseek_ocr__score_f1',
       'qwen2_5_vl_3b__sample_score', 'qwen2_5_vl_3b__cost',
       'qwen2_5_vl_3b__valid_mask', 'qwen2_5_vl_3b__is_correct',
       'qwen2_5_vl_3b__score_f1', 'qwen2_5_vl_7b__sample_score',
       'qwen2_5_vl_7b__cost', 'qwen2_5_vl_7b__valid_mask',
       'qwen2_

In [20]:
# Router columns
ROUTER_BEST_MODEL_COL = "router_best_model_name"
ROUTER_PERF_COL       = "router_chosen_perf"
ROUTER_COST_COL       = "router_chosen_cost"



for col in [ROUTER_CORRECT_COL, ROUTER_COST_COL]:
    assert col in df_test.columns, f"Missing {col} in test df"


### Step-3: - Fetch Model pricing

In [16]:
# The models in this test set
MODEL_NAMES = [
    "deepseek_ocr",
    "qwen2_5_vl_3b",
    "qwen2_5_vl_7b",
    "qwen3_vl_8b_thinking",
    "gemma_3_27b",
]

MODEL_NAMES

['deepseek_ocr',
 'qwen2_5_vl_3b',
 'qwen2_5_vl_7b',
 'qwen3_vl_8b_thinking',
 'gemma_3_27b']

In [17]:
required_suffixes = ["__is_correct", "__cost", "__valid_mask", "__score_f1", "__sample_score"]

missing_cols = []
for name in MODEL_NAMES:
    for suf in required_suffixes:
        col = f"{name}{suf}"
        if col not in df_test.columns:
            missing_cols.append(col)

if missing_cols:
    print("WARNING: missing columns (check if you really need all of them):")
    print(missing_cols)
else:
    print("All expected model columns are present.")


All expected model columns are present.


In [21]:
def compute_router_is_correct(df, model_col, out_col="router_is_correct"):
    """
    For each row:
      - read router_best_model_name
      - pick that model's `<name>__is_correct` column
      - write boolean (0/1) into `out_col`
    """
    is_corr = []

    for idx, row in df.iterrows():
        m_name = row[model_col]
        if pd.isna(m_name):
            is_corr.append(0.0)
            continue

        col_corr = f"{m_name}__is_correct"
        if col_corr not in df.columns:
            # If something is off, treat as incorrect
            is_corr.append(0.0)
            continue

        val = row[col_corr]
        # cast to float 0/1
        if pd.isna(val):
            is_corr.append(0.0)
        else:
            is_corr.append(float(val))

    df[out_col] = is_corr
    return df

df_test = compute_router_is_correct(df_test, ROUTER_BEST_MODEL_COL)
df_test[["router_task", ROUTER_BEST_MODEL_COL, "router_is_correct", ROUTER_COST_COL]].head()


,router_task,router_best_model_name,router_is_correct,router_chosen_cost
0,diagram_reasoning,gemma_3_27b,1.0,0.000060
1,diagram_reasoning,gemma_3_27b,0.0,0.000042
2,diagram_reasoning,qwen2_5_vl_3b,0.0,0.000020
3,diagram_reasoning,gemma_3_27b,1.0,0.000050
4,diagram_reasoning,qwen2_5_vl_3b,1.0,0.000028


In [22]:
def summarize_model_on_subset(sub_df: pd.DataFrame, model_name: str):
    """
    For a given (task subset, model), compute:
      - num_samples: total rows in this task
      - num_valid: rows where model is valid
      - accuracy: mean(is_correct) over valid rows
      - avg_cost_per_sample: mean(cost) over valid rows
      - total_cost: sum(cost) over valid rows
    """
    col_corr  = f"{model_name}__is_correct"
    col_cost  = f"{model_name}__cost"
    col_valid = f"{model_name}__valid_mask"

    if col_corr not in sub_df.columns:
        raise ValueError(f"Missing {col_corr} in dataframe for model {model_name}")
    if col_cost not in sub_df.columns:
        raise ValueError(f"Missing {col_cost} in dataframe for model {model_name}")
    if col_valid not in sub_df.columns:
        # If no valid mask, assume all valid
        valid_mask = pd.Series(True, index=sub_df.index)
    else:
        valid_mask = sub_df[col_valid].fillna(False).astype(bool)

    valid_df = sub_df[valid_mask]
    num_samples = len(sub_df)
    num_valid = len(valid_df)

    if num_valid == 0:
        return {
            "model_name": model_name,
            "num_samples": num_samples,
            "num_valid": num_valid,
            "accuracy": np.nan,
            "avg_cost_per_sample": np.nan,
            "total_cost": np.nan,
        }

    corr = valid_df[col_corr].fillna(0.0).astype(float)
    cost = valid_df[col_cost].astype(float)

    acc = float(corr.mean())
    avg_cost = float(cost.mean())
    total_cost = float(cost.sum())

    return {
        "model_name": model_name,
        "num_samples": num_samples,
        "num_valid": num_valid,
        "accuracy": acc,
        "avg_cost_per_sample": avg_cost,
        "total_cost": total_cost,
    }


In [23]:
summary_rows = []

for task, task_df in df_test.groupby(TASK_COL):
    for name in MODEL_NAMES:
        s = summarize_model_on_subset(task_df, name)
        s["task"] = task
        summary_rows.append(s)

model_task_perf_test = pd.DataFrame(summary_rows)

# nicer ordering
model_task_perf_test = model_task_perf_test[
    ["task", "model_name", "num_samples", "num_valid", "accuracy",
     "avg_cost_per_sample", "total_cost"]
].sort_values(["task", "accuracy"], ascending=[True, False])

model_task_perf_test.head(15)


,task,model_name,num_samples,num_valid,accuracy,avg_cost_per_sample,total_cost
3,abstract_reasoning,qwen3_vl_8b_thinking,284,284,0.619718,0.001185,0.336451
1,abstract_reasoning,qwen2_5_vl_3b,284,284,0.605634,0.000080,0.022839
4,abstract_reasoning,gemma_3_27b,284,284,0.605634,0.000063,0.017870
2,abstract_reasoning,qwen2_5_vl_7b,284,284,0.566901,0.000191,0.054316
0,abstract_reasoning,deepseek_ocr,284,277,0.472924,0.000029,0.007906
6,chart_captioning,qwen2_5_vl_3b,602,602,0.634551,0.000076,0.045836
7,chart_captioning,qwen2_5_vl_7b,602,602,0.116279,0.000187,0.112794
5,chart_captioning,deepseek_ocr,602,602,0.028239,0.000038,0.022617
9,chart_captioning,gemma_3_27b,602,602,0.003322,0.000077,0.046109
8,chart_captioning,qwen3_vl_8b_thinking,602,602,0.001661,0.001151,0.693050


In [24]:
idx_best = model_task_perf_test.groupby("task")["accuracy"].idxmax()
lovm_test_baseline = model_task_perf_test.loc[idx_best].copy()

lovm_test_baseline = lovm_test_baseline.sort_values("task").reset_index(drop=True)
lovm_test_baseline


,task,model_name,num_samples,num_valid,accuracy,avg_cost_per_sample,total_cost
0,abstract_reasoning,qwen3_vl_8b_thinking,284,284,0.619718,0.001185,0.336451
1,chart_captioning,qwen2_5_vl_3b,602,602,0.634551,0.000076,0.045836
2,chart_reasoning,qwen2_5_vl_7b,1277,1277,0.898982,0.000100,0.128315
3,code_generation,qwen2_5_vl_3b,317,317,0.022082,0.000059,0.018800
4,counting,qwen2_5_vl_7b,283,283,0.819788,0.000066,0.018788
...,...,...,...,...,...,...,...
25,table_reasoning,qwen2_5_vl_7b,1822,1822,0.672338,0.000126,0.229342
26,textbook_qa,qwen3_vl_8b_thinking,216,216,0.222222,0.000547,0.118059
27,ui_captioning,qwen2_5_vl_3b,281,281,0.074733,0.000237,0.066602
28,visual_mrc,qwen2_5_vl_7b,302,302,0.350993,0.001221,0.368760


In [25]:
# Map task → chosen model_name
task_to_lovm_model = dict(zip(lovm_test_baseline["task"], lovm_test_baseline["model_name"]))

lovm_model_for_sample = []
lovm_is_correct = []
lovm_cost = []

for _, row in df_test.iterrows():
    task = row[TASK_COL]
    m_name = task_to_lovm_model[task]

    col_corr = f"{m_name}__is_correct"
    col_cost = f"{m_name}__cost"

    # correctness
    corr_val = row.get(col_corr, 0.0)
    if pd.isna(corr_val):
        corr_val = 0.0
    lovm_is_correct.append(float(corr_val))

    # cost
    cost_val = row.get(col_cost, 0.0)
    if pd.isna(cost_val):
        cost_val = 0.0
    lovm_cost.append(float(cost_val))

    lovm_model_for_sample.append(m_name)

df_test["lovm_model_name"] = lovm_model_for_sample
df_test["lovm_is_correct"] = lovm_is_correct
df_test["lovm_cost"] = lovm_cost

df_test[[TASK_COL, "lovm_model_name", "lovm_is_correct", "lovm_cost"]].head()


,router_task,lovm_model_name,lovm_is_correct,lovm_cost
0,diagram_reasoning,qwen2_5_vl_7b,1.0,0.000163
1,diagram_reasoning,qwen2_5_vl_7b,1.0,0.000531
2,diagram_reasoning,qwen2_5_vl_7b,0.0,0.000041
3,diagram_reasoning,qwen2_5_vl_7b,1.0,0.000335
4,diagram_reasoning,qwen2_5_vl_7b,1.0,0.000056


In [26]:
overall_lovm_acc   = df_test["lovm_is_correct"].mean()
overall_lovm_cost  = df_test["lovm_cost"].mean()
overall_lovm_total = df_test["lovm_cost"].sum()

overall_router_acc   = df_test["router_is_correct"].mean()
overall_router_cost  = df_test[ROUTER_COST_COL].mean()
overall_router_total = df_test[ROUTER_COST_COL].sum()

overall_summary_test = pd.DataFrame({
    "method": ["lovm_baseline", "router"],
    "accuracy": [overall_lovm_acc, overall_router_acc],
    "avg_cost_per_sample": [overall_lovm_cost, overall_router_cost],
    "total_cost": [overall_lovm_total, overall_router_total],
})

overall_summary_test


,method,accuracy,avg_cost_per_sample,total_cost
0,lovm_baseline,0.651492,0.000250,3.425495
1,router,0.646604,0.000036,0.498103


In [30]:
# Router per-task summary on test
router_task_summary_test = (
    df_test
    .groupby(TASK_COL)
    .agg(
        num_samples_router = ("router_is_correct", "size"),
        accuracy_router = ("router_is_correct", "mean"),
        avg_cost_per_sample_router = (ROUTER_COST_COL, "mean"),
        total_cost_router = (ROUTER_COST_COL, "sum"),
    )
    .reset_index()
    .rename(columns={TASK_COL: "task"})
)

# Merge with lovm per-task stats
per_task_comparison_test = lovm_test_baseline.merge(
    router_task_summary_test,
    on="task",
    how="inner",
)

# Reorder columns for readability
per_task_comparison_test = per_task_comparison_test[[
    "task",
    "model_name",                # LOVM baseline chosen model
    "num_samples", "num_valid",  # from lovm_test_baseline
    "accuracy", "avg_cost_per_sample", "total_cost",  # LOVM-style baseline
    "num_samples_router",        # from router summary
    "accuracy_router", "avg_cost_per_sample_router", "total_cost_router",
]]

per_task_comparison_test.sort_values("task")


,task,model_name,num_samples,num_valid,accuracy,avg_cost_per_sample,total_cost,num_samples_router,accuracy_router,avg_cost_per_sample_router,total_cost_router
0,abstract_reasoning,qwen3_vl_8b_thinking,284,284,0.619718,0.001185,0.336451,284,0.919014,0.000044,0.012442
1,chart_captioning,qwen2_5_vl_3b,602,602,0.634551,0.000076,0.045836,602,0.611296,0.000060,0.036307
2,chart_reasoning,qwen2_5_vl_7b,1277,1277,0.898982,0.000100,0.128315,1277,0.910728,0.000032,0.041194
3,code_generation,qwen2_5_vl_3b,317,317,0.022082,0.000059,0.018800,317,0.022082,0.000042,0.013343
4,counting,qwen2_5_vl_7b,283,283,0.819788,0.000066,0.018788,283,0.890459,0.000019,0.005439
...,...,...,...,...,...,...,...,...,...,...,...
25,table_reasoning,qwen2_5_vl_7b,1822,1822,0.672338,0.000126,0.229342,1822,0.667398,0.000043,0.078779
26,textbook_qa,qwen3_vl_8b_thinking,216,216,0.222222,0.000547,0.118059,216,0.083333,0.000030,0.006455
27,ui_captioning,qwen2_5_vl_3b,281,281,0.074733,0.000237,0.066602,281,0.014235,0.000037,0.010307
28,visual_mrc,qwen2_5_vl_7b,302,302,0.350993,0.001221,0.368760,302,0.261589,0.000024,0.007306


In [40]:
import plotly.express as px

# Build a long-form dataframe for Plotly: one row per (task, method)
plot_rows = []
for _, r in per_task_comparison_test.iterrows():
    # LOVM-style baseline
    plot_rows.append({
        "task": r["task"],
        "method": "LOVM baseline",
        "accuracy": r["accuracy"],
        "avg_cost_per_sample": r["avg_cost_per_sample"],
        "total_cost": r["total_cost"],
        "num_samples": r["num_samples"],
    })
    # Router
    plot_rows.append({
        "task": r["task"],
        "method": "Router",
        "accuracy": r["accuracy_router"],
        "avg_cost_per_sample": r["avg_cost_per_sample_router"],
        "total_cost": r["total_cost_router"],
        "num_samples": r["num_samples_router"],
    })

df_plot = pd.DataFrame(plot_rows)


In [45]:

# Optional: use log-scale on x to spread out small costs
fig = px.scatter(
    df_plot,
    x="avg_cost_per_sample",
    y="accuracy",
    color="method",
    symbol="method",
    hover_name="task",
    hover_data={
        "task": True,
        "method": True,
        "avg_cost_per_sample": ":.6f",
        "accuracy": ":.3f",
        "total_cost": ":.3f",
        "num_samples": True,
    },
    labels={
        "avg_cost_per_sample": "Avg cost per sample ($)",
        "accuracy": "Accuracy",
    },
    title="Per-task Accuracy vs Cost (Test Set)",
    log_x=True,               # set to False if you prefer linear scale
    template="plotly_white",
)

# Make markers a bit larger and improve layout
fig.update_traces(marker=dict(size=9, line=dict(width=1)))
fig.update_layout(
    title="Per-task Accuracy vs Cost (Test Set)",
    xaxis_title="Avg cost per sample ($, log scale)",
    yaxis_title="Accuracy",
    template="plotly_white",
    legend_title_text="Method",
    xaxis=dict(
        type="log",              # ← ENABLE LOG SCALE
        # tickformat=".2e",        # ← CLEAN SCIENTIFIC NOTATION (e.g., 1e-4)
        tickfont=dict(size=10),
        showgrid=True,
    ),
    yaxis=dict(range=[0, 1.05]),
    hovermode="closest",
)

fig.show()
